# 17 - Conformal Prediction Analysis

This notebook evaluates the predictive uncertainty of the final Full Multimodal XGBoost classifier using Conformal Prediction.

Unlike conventional machine learning models that return a single predicted class with associated probabilities, conformal prediction generates statistically valid prediction sets that quantify prediction uncertainty at a predefined confidence level. This provides an additional layer of reliability by indicating whether multiple diagnostic classes should be considered for a given participant.

The conformal predictor is calibrated using the independent participant-level validation dataset and evaluated on the participant-level test dataset without retraining the original classifier.

The following aspects are investigated:

- Prediction set coverage
- Average prediction set size
- Singleton prediction rate
- Ambiguous prediction rate
- Representative prediction set examples

This analysis complements the previous performance evaluation and Explainable Artificial Intelligence (SHAP) analyses by providing statistically grounded uncertainty estimates for the proposed Parkinson's disease screening framework.

## Importing Libraries and Setting Paths

In [4]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mapie.classification import SplitConformalClassifier
# PROJECT PATHS
ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "outputs"
CONFORMAL_DIR = OUTPUT_DIR / "conformal"
TABLE_DIR = CONFORMAL_DIR / "tables"
FIGURE_DIR = CONFORMAL_DIR / "figures"
TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
print("Project root:", ROOT)

Project root: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease


## Loading the Final Model and Datasets

The final Full Multimodal XGBoost classifier is loaded together with the corresponding participant-level validation and independent test datasets.

The validation dataset is reserved for conformal calibration, while the independent participant-level test dataset is used exclusively to evaluate the resulting prediction sets.

Before calibration, the feature names expected by the trained classifier are verified against both datasets to ensure complete consistency.

In [5]:
# MODEL AND DATASET PATHS
MODEL_PATH = (
    MODEL_DIR /
    "full_multimodal_xgboost.joblib"
)

VALIDATION_PATH = (
    DATA_DIR /
    "validation_multimodal_full_task_aware.csv"
)

TEST_PATH = (
    DATA_DIR /
    "test_multimodal_full_task_aware.csv"
)

In [ ]:
# LOAD MODEL
classifier = joblib.load(MODEL_PATH)
print(type(classifier))

<class 'sklearn.pipeline.Pipeline'>


In [7]:
# LOAD DATASETS
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Validation: (70, 2412)
Test: (71, 2412)


In [8]:
# VERIFY FEATURES
feature_names = list(
    classifier.feature_names_in_
)

missing_validation = [
    feature
    for feature in feature_names
    if feature not in validation_df.columns
]

missing_test = [
    feature
    for feature in feature_names
    if feature not in test_df.columns
]

print("Expected features:", len(feature_names))
print("Missing validation:", len(missing_validation))
print("Missing test:", len(missing_test))

Expected features: 2405
Missing validation: 0
Missing test: 0


In [9]:
# PREPARE DATA
X_validation = validation_df[
    feature_names
].copy()

y_validation = validation_df[
    "label"
].copy()

X_test = test_df[
    feature_names
].copy()

y_test = test_df[
    "label"
].copy()

print("Validation features:", X_validation.shape)
print("Test features:", X_test.shape)

print("Validation classes:", sorted(y_validation.unique()))
print("Test classes:", sorted(y_test.unique()))

Validation features: (70, 2405)
Test features: (71, 2405)
Validation classes: [np.int64(0), np.int64(1), np.int64(2)]
Test classes: [np.int64(0), np.int64(1), np.int64(2)]
